# Attention Visualized

This notebook builds a tiny attention layer and lets you watch
it in action. You will see which words attend to which other
words and how the causal mask blocks the future. Every number
is real. Every heatmap shows actual attention weights.

We will feed the sentence *The cat sat on the mat* through a
single attention head and plot the weight matrix as a heatmap.
Then we will add more heads and see how each one specializes.

No training needed. The weights are random. But the patterns
are real. The causal mask always blocks the future. The softmax
always makes each row sum to one.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

## Build a tiny attention module

This is the exact same attention code from the guide but
modified to return the attention weights for visualization.

In [ ]:
class SingleHeadAttention(nn.Module):
    def __init__(self, d_model, head_dim):
        super().__init__()
        self.head_dim = head_dim
        self.w_q = nn.Linear(d_model, head_dim, bias=False)
        self.w_k = nn.Linear(d_model, head_dim, bias=False)
        self.w_v = nn.Linear(d_model, head_dim, bias=False)

    def forward(self, x, causal=True, return_weights=False):
        batch, seq_len, d_model = x.shape
        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if causal:
            mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
            scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)
        output = weights @ v

        if return_weights:
            return output, weights
        return output


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, causal=True, return_weights=False):
        batch, seq_len, d_model = x.shape
        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if causal:
            mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
            mask = mask.view(1, 1, seq_len, seq_len)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)
        output = weights @ v
        output = output.transpose(1, 2).contiguous()
        output = output.reshape(batch, seq_len, d_model)
        output = self.out_proj(output)

        if return_weights:
            return output, weights
        return output

## Create a tiny embedding for our sentence

We will use small random vectors instead of a real embedding
table. The attention patterns still work the same way.

In [ ]:
torch.manual_seed(42)
words = ["The", "cat", "sat", "on", "the", "mat"]
seq_len = len(words)

d_model = 768
x = torch.randn(1, seq_len, d_model)

print(f"Sequence length: {seq_len}")
print(f"Input shape: {x.shape}")
print(f"Words: {words}")

## Single head attention heatmap

One attention head produces one weight matrix. Each row is
a token acting as query. Each column is a token acting as key.
The color shows how much attention token i gives to token j.

In [ ]:
attn_single = SingleHeadAttention(d_model, head_dim=64)
_, weights = attn_single(x, causal=True, return_weights=True)

w = weights[0].detach().numpy()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(w, cmap='YlOrRd', vmin=0, vmax=1)

ax.set_xticks(range(seq_len))
ax.set_xticklabels(words, rotation=45, ha='right')
ax.set_yticks(range(seq_len))
ax.set_yticklabels(words)
ax.set_xlabel('Key (what I offer)')
ax.set_ylabel('Query (what I am looking for)')
ax.set_title('Single Head Attention Weights')

for i in range(seq_len):
    for j in range(seq_len):
        color = 'white' if w[i, j] > 0.5 else 'black'
        ax.text(j, i, f'{w[i, j]:.2f}', ha='center', va='center', color=color, fontsize=9)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

print("Upper right is all zeros.")
print("The causal mask blocks tokens from seeing the future.")
print(f"Token 0 sees only itself: {w[0, 0]:.2f}")
print(f"Token 5 sees all six tokens: max={w[5].max():.2f}")

## What happens without the causal mask

If we turn off the mask every token can see every other token.
This is what an encoder like BERT uses. But a GPT cannot do
this during training or it would cheat.

In [ ]:
_, weights_no_mask = attn_single(x, causal=False, return_weights=True)

w = weights_no_mask[0].detach().numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

_, weights_causal = attn_single(x, causal=True, return_weights=True)
w_causal = weights_causal[0].detach().numpy()

ax1.imshow(w_causal, cmap='YlOrRd', vmin=0, vmax=1)
ax1.set_xticks(range(seq_len))
ax1.set_xticklabels(words, rotation=45, ha='right')
ax1.set_yticks(range(seq_len))
ax1.set_yticklabels(words)
ax1.set_title('With causal mask (GPT)')

ax2.imshow(w, cmap='YlOrRd', vmin=0, vmax=1)
ax2.set_xticks(range(seq_len))
ax2.set_xticklabels(words, rotation=45, ha='right')
ax2.set_yticks(range(seq_len))
ax2.set_yticklabels(words)
ax2.set_title('Without causal mask (no cheating prevention)')

plt.tight_layout()
plt.show()

## Multi-head attention: twelve heads

Each head learns a different pattern. Even with random weights
you can see that each head produces a different weight matrix.
In a trained model these differences would correspond to actual
linguistic patterns like subject-verb agreement.

In [ ]:
attn_multi = MultiHeadAttention(d_model=768, num_heads=12)
_, weights = attn_multi(x, causal=True, return_weights=True)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for h in range(12):
    w = weights[0, h].detach().numpy()
    ax = axes[h]
    im = ax.imshow(w, cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_xticks(range(seq_len))
    ax.set_xticklabels(words, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(seq_len))
    ax.set_yticklabels(words, fontsize=7)
    ax.set_title(f'Head {h + 1}', fontsize=10)

plt.suptitle('All 12 Attention Heads in One Transformer Layer', fontsize=14)
plt.tight_layout()
plt.show()

print("Each head sees the same input but produces different attention patterns.")
print("The causal mask is always present. Every head has the upper right blocked.")

## Try your own sentence

Change the words below and rerun the cells to see how attention
behaves with different inputs. The patterns will change because
each word's random embedding is different. With real trained
embeddings you would see meaningful linguistic patterns.

In [ ]:
your_words = ["The", "dog", "ran", "through", "the", "park"]

x2 = torch.randn(1, len(your_words), 768)
_, weights = attn_multi(x2, causal=True, return_weights=True)

fig, axes = plt.subplots(2, 6, figsize=(20, 7))
axes = axes.flatten()

for h in range(12):
    w = weights[0, h].detach().numpy()
    ax = axes[h]
    ax.imshow(w, cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_xticks(range(len(your_words)))
    ax.set_xticklabels(your_words, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(len(your_words)))
    ax.set_yticklabels(your_words, fontsize=7)
    ax.set_title(f'Head {h + 1}', fontsize=9)

plt.suptitle(f'Attention patterns for: {" ".join(your_words)}', fontsize=14)
plt.tight_layout()
plt.show()

## What these heatmaps teach you

The upper right triangle is always empty. The causal mask blocks
every token from seeing words that come after it. This is what
makes GPT a language model instead of an encoder.

The diagonal is often bright. Every token attends to itself.
Self attention is important because no other token has exactly
the same information you have about yourself.

The first row is always solid on column zero. Token zero has
only itself to look at. It must put all its attention there.

In a trained model the attention patterns would be meaningful.
Head three might connect subjects to their verbs. Head seven
might connect pronouns to the nouns they refer to. With random
weights the patterns are random but the structure is clear.
The model has the capacity to learn these patterns. It just
needs data and training.